# Scaling Infrastructure: Autoscaling, Distributed Sharding & Multi-Tier Caching

Operating planetary-scale ML infrastructure demands elasticity and distributed memory management:
1. **Cluster Autoscaling (HPA / KEDA Pattern)**: Dynamic replica provisioning based on real-time concurrency with anti-flapping cooldowns.
2. **Distributed Model Sharding**: Sizing memory and partitioning large foundation models across Tensor Parallelism (TP), Pipeline Parallelism (PP), and ZeRO stages.
3. **Multi-Tier Caching**: Combining ultra-fast local in-process memory (L1) with distributed Redis clusters (L2) to absorb redundant traffic.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('08-system-design/scaling-infrastructure/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import cluster_sim
print('cluster_sim loaded')

## 1. Kubernetes Autoscaler with Anti-Flapping Cooldowns
Preventing rapid pod churn while scaling up dynamically under traffic spikes.

In [ ]:
from cluster_sim import ClusterAutoScaler

scaler = ClusterAutoScaler(
    min_replicas=2,
    max_replicas=10,
    target_concurrency_per_replica=10.0,
    scale_up_cooldown_s=15.0,
    scale_down_cooldown_s=45.0
)

# Initial steady-state traffic (20 concurrent users)
d1 = scaler.evaluate(current_time=0.0, active_load=20.0)
print(f"t=0s  : Replicas={d1.current_replicas} (Desired={d1.desired_replicas}) | Reason: {d1.reason}")

# Sudden traffic surge (65 concurrent users)
d2 = scaler.evaluate(current_time=1.0, active_load=65.0)
print(f"t=1s  : Scaled to {d2.current_replicas} Pods! | Reason: {d2.reason}")

# Another spike immediately at t=5s -> blocked by scale-up cooldown
d3 = scaler.evaluate(current_time=5.0, active_load=95.0)
print(f"t=5s  : Scaling blocked? {not d3.scaled} | Reason: {d3.reason}")

# Traffic drops back to 20 users at t=20s -> scale-down cooldown prevents premature termination
d4 = scaler.evaluate(current_time=20.0, active_load=20.0)
print(f"t=20s : Scaling blocked? {not d4.scaled} | Reason: {d4.reason}")

## 2. Distributed Model Sharding Budget Calculator
Calculating per-GPU memory consumption and determining if a foundation model fits on target hardware.

In [ ]:
from cluster_sim import calculate_distributed_memory

# 1. 70-Billion parameter model (e.g. LLaMA-3 70B in FP16)
# Try on single 80GB GPU
single_gpu = calculate_distributed_memory(param_count_billions=70.0, precision_bytes=2, tp_degree=1, gpu_memory_gb=80.0)
print(f"70B on 1x 80GB GPU: Fits? {single_gpu.fits_in_memory} ({single_gpu.notes})")

# Try with Tensor Parallelism TP=4
tp4_gpu = calculate_distributed_memory(param_count_billions=70.0, precision_bytes=2, tp_degree=4, gpu_memory_gb=80.0)
print(f"70B on 4x 80GB GPUs (TP=4): Fits? {tp4_gpu.fits_in_memory} ({tp4_gpu.notes})")

# 2. 405-Billion parameter model with TP=8 and PP=2
huge_model = calculate_distributed_memory(param_count_billions=405.0, precision_bytes=2, tp_degree=8, pp_degree=2, gpu_memory_gb=80.0)
print(f"405B on 16x 80GB GPUs (TP=8, PP=2): Fits? {huge_model.fits_in_memory} ({huge_model.notes})")

## 3. Multi-Tier Cache (L1 In-Memory + L2 Distributed Redis)
Reducing redundant embedding and feature store requests.

In [ ]:
from cluster_sim import MultiTierCache

cache = MultiTierCache(l1_max_items=3)

# Populate cache
cache.put("user_100_features", {"tenure": 12, "tier": "gold"})

# Step 1: Hit in L1 memory
val1, status1 = cache.get("user_100_features")
print(f"Read 1: Status={status1}, Value={val1}")

# Step 2: Flood L1 to trigger eviction to L2
cache.put("k1", "v1")
cache.put("k2", "v2")
cache.put("k3", "v3")

# Step 3: Hit in L2 distributed cache and warm back into L1
val2, status2 = cache.get("user_100_features")
print(f"Read 2: Status={status2}, Value={val2}")
print(f"Cache Statistics: {cache.stats}")